# 11 — LLM + RAG Layer: the AI Analyst

Phase 11 of the Indian Equity AI project plan.

Builds a retrieval index over this project's own real, already-computed evidence
(fundamentals, technical snapshots, news+sentiment, model predictions, SHAP
explanations), tagged with an honest source-reliability tier, and answers example
AI Analyst questions (Section 35) with cited, grounded evidence.

**The "LLM" here is a deterministic template composer, not a generative model call**
(see PROJECT_PLAN.md Phase 11 for why) -- it can only restate retrieved evidence plus
its source, so it cannot invent facts or predictions that aren't already in the corpus.

In [1]:
# torch must be imported before pandas/pyarrow anywhere in this process
# (sentence-transformers depends on torch) -- see src/news/sentiment.py.
import torch  # noqa: F401

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.explainability.shap_explainer import top_factors
from src.rag.documents import (
    build_explanation_documents,
    build_fundamentals_documents,
    build_news_documents,
    build_risk_documents,
    build_technical_documents,
)
from src.rag.retrieval import VectorIndex, load_sentence_transformer_embedder
from src.rag.analyst import answer_query

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
UNIVERSE = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]

## Step 1: Build the document corpus from Phases 4/6/8/10's real outputs

In [3]:
fundamentals_snapshot = pd.read_csv(DATA_PROCESSED / "fundamentals_snapshot.csv", index_col=0)
news_scored = pd.read_csv(DATA_PROCESSED / "news_scored.csv")
risk_output = pd.read_parquet(DATA_PROCESSED / "risk_engine_output.parquet")
explainability_predictions = pd.read_parquet(DATA_PROCESSED / "explainability_predictions.parquet")
shap_values = pd.read_parquet(DATA_PROCESSED / "explainability_shap_values.parquet")

technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
technical_latest = technical.sort_values("date").groupby("symbol").tail(1)

documents = (
    build_fundamentals_documents(fundamentals_snapshot)
    + build_technical_documents(technical_latest)
    + build_news_documents(news_scored)
    + build_risk_documents(risk_output)
    + build_explanation_documents(explainability_predictions, shap_values, top_factors, n_factors=3)
)

print("Total documents:", len(documents))
pd.DataFrame(documents)["category"].value_counts()

Total documents: 32


category
news                12
fundamentals         5
technical            5
model_prediction     5
explanation          5
Name: count, dtype: int64

## Step 2: Build the vector index (real sentence-transformers embeddings)

In [4]:
embed_fn = load_sentence_transformer_embedder()
index = VectorIndex(embed_fn)
index.build(documents)
print("Index built with", len(index.documents), "documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built with 32 documents.


## Step 3: Example AI Analyst queries (project plan Section 35)

In [5]:
example_queries = [
    ("RELIANCE", "Why is this stock bullish or bearish right now?"),
    ("RELIANCE", "What are the major risks?"),
    ("ITC", "What are the strongest fundamental factors?"),
    ("TCS", "How has recent news affected this stock?"),
]

for symbol, query in example_queries:
    result = answer_query(query, symbol, index, k=4)
    print(f"\n{'='*70}\nQ ({symbol}): {query}\n{'='*70}")
    print(result["answer"])


Q (RELIANCE): Why is this stock bullish or bearish right now?
Based on the retrieved evidence for RELIANCE:
1. RELIANCE prediction explanation on 2026-09-24: probability of up move 47.9%, confidence 0.04 (Low). Top factors pushing bullish (log-odds units): volatility_30d (+0.178), volume_ratio (+0.100), stoch_d (+0.070). Top factors pushing bearish (log-odds units): stoch_k (-0.128), return_60d (-0.072), rsi_14 (-0.069). (Source: This project's own ML pipeline (Phase 10, SHAP), Internal model output (not an independent source))
2. RELIANCE news (2026-07-25 01:00:25+00:00): "Reliance Industries Ltd (WBO:RLI) Q1 2027 Earnings Call Highlights: Strong Revenue Growth Amid ...". FinBERT sentiment: positive (positive=0.96, negative=0.02). Event type(s): earnings. (Source: GuruFocus.com, Tier 3 (financial website/aggregator))
3. RELIANCE news (2026-09-18 10:12:51+00:00): "Reliance (NYSE:RS) Gets A Lift From Tax Relief As Valuation Looks Nearly Fair". FinBERT sentiment: positive (positive=0.92

## Step 4: Verify source-tier trust weighting actually changes ranking

(A sanity check on Section 33's requirement -- higher-tier sources should be
preferred when relevance is comparable.)

In [6]:
raw_results = index.search("latest news", k=10, symbol="RELIANCE")
pd.DataFrame(raw_results)[["doc_id", "category", "tier_label", "similarity", "score"]]

,doc_id,category,tier_label,similarity,score
0,news_RELIANCE_7b1303e6-6df8-3cbb-b0e5-52172653...,news,Tier 2 (major wire/publication),0.312642,0.281378
1,news_RELIANCE_7408502c-d216-344d-921b-d6dd8859...,news,Tier 3 (financial website/aggregator),0.325476,0.260381
2,news_RELIANCE_037ad3da-442c-3d2d-a882-51adf0c9...,news,Tier 3 (financial website/aggregator),0.305006,0.244005
3,news_RELIANCE_d0ec11ad-056a-3ece-b6d5-86c3dbf6...,news,Tier 3 (financial website/aggregator),0.288364,0.230691
4,news_RELIANCE_30bad952-ef64-3c62-b904-7f4ce594...,news,Tier 2 (major wire/publication),0.250564,0.225507
5,fund_RELIANCE,fundamentals,Tier 3 (financial website/aggregator),0.280947,0.224758
6,risk_RELIANCE_2026-09-18,model_prediction,Internal model output (not an independent source),0.181854,0.154576
7,explain_RELIANCE_2026-09-24,explanation,Internal model output (not an independent source),0.160515,0.136437
8,tech_RELIANCE_2026-09-25,technical,Tier 3 (financial website/aggregator),0.125000,0.100000


## Reflection

- Every answer above is composed entirely from retrieved, cited evidence -- restating
  each document's own text plus its source and tier. Since the composer is a template,
  not a generative model, it structurally cannot invent a fact, number, or prediction
  that isn't already in the corpus (project plan Section 32's core requirement).
- **Honest tiering, not inflated:** none of this project's data is tagged Tier 1
  (official NSE/BSE/SEBI/filings) because none of it was fetched directly from those
  sources -- everything here is Yahoo-Finance-relayed (Tier 3) or this project's own
  model output (a distinct, always-labeled internal category). A production system
  wanting genuine Tier 1 coverage would need direct NSE/BSE/SEBI/company-filing
  ingestion (see Indian_Equity_AI_Project_Plan.md Section 5's licensing notes).
- **Bug found and fixed while building this phase:** Phase 10 saved
  `explainability_predictions.parquet` with a fresh reset index but
  `explainability_shap_values.parquet` with its original pre-reset index --
  reloading them separately (as this notebook does) would have silently misaligned
  every row. Fixed by resetting both to a shared index before saving in Phase 10.
- **What a real LLM would add later:** more natural free-text answers and better
  handling of queries the retrieval step doesn't match well -- but only behind a strict
  "cite only what's retrieved" system prompt, since a generative model without that
  discipline reintroduces exactly the hallucination risk this template approach avoids
  by construction. The `answer_query` interface is intentionally decoupled from *how*
  answers are composed, so swapping in a real LLM later doesn't require touching the
  retrieval/document/tiering infrastructure built in this phase.